# **Multiple Imputation, RobustScaler Nromalization, PCA dimensionality reduction, and XGBoost**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split

# LOADING AND EXPLORING THE DATASET

In [ ]:
df = pd.read_excel('salesworkload.xlsx', index_col=None, skiprows=1)

In [ ]:
df.head()

# Data Type Modification

In [ ]:
df.info()

* **MonthYear:** Date in month and year 
* **Time Index:** The period at which the total sales occur
* **Country:** Name of country of the retail industry
* **StoreID:** The given id for specific store
* **City:** Name of city of the retail industry
* **Dept_ID:** The given id for the sales department
* **Dept. Name:** Name of the sales department
* **HoursOwn:** Total hours worked
* **HoursLease:** Total hours of operation rented
* **Sales units:** Total amount of sales
* **Turnover:** The rate at which goods are sold and replaced in a shop.
* **Customer:** The recipient of a good, service, product or an idea - obtained from a seller, vendor, or supplier
* **Area (m2):** Total working are of the retail industry
* **Opening hours:** normal business hours

variables like 'Time index', 'Country', 'StoreID', 'City', 'Dept_ID','Dept. Name', 'Opening hours' have to be categorical and HoursOwn, HoursLease, Sales units, Turnover, Area (m2) have to be numerical types, so data type conversion is required. In addition, the MonthYear should be of type datetime, so it needs to be converted. 

In [ ]:
HoursOwn_notnum = df[['HoursOwn']].applymap(np.isreal).values
df[~HoursOwn_notnum][['Dept. Name', 'HoursOwn']]

In [ ]:
# Calculate the mean of HoursOwn for the specified department and without the cells containing a question mark
imp_dept_x = df.loc[(df['Dept. Name'] == 'Meat') & (df['HoursOwn'] != '?')]['HoursOwn'].mean()
imp_dept_y = df.loc[(df['Dept. Name'] == 'Frozen') & (df['HoursOwn'] != '?')]['HoursOwn'].mean()

# Replace the question marks
df.loc[(df['Dept. Name'] == 'Meat') & (df['HoursOwn'] == '?'), ['HoursOwn']] = imp_dept_x
df.loc[(df['Dept. Name'] == 'Frozen') & (df['HoursOwn'] == '?'), ['HoursOwn']] = imp_dept_y

# Give an information about the new values
print('Department Meat: ' + str(imp_dept_x))
print('Department Frozen: ' + str(imp_dept_y))

# convert column now to numeric
df['HoursOwn'] = pd.to_numeric(df['HoursOwn'])

In [ ]:
Area_notnum = df[['Area (m2)']].applymap(np.isreal).values
df[~Area_notnum][['Dept. Name', 'Area (m2)']]

In [ ]:
# Calculate the mean of Area for the specified department and without the cells containing a question mark
imp_dept = df.loc[(df['Area (m2)'] != '#NV')]['Area (m2)'].mean()

# Replace the question marks
df.loc[(df['Area (m2)'] == '#NV'), ['Area (m2)']] = imp_dept

# Give an information about the new values
print('mean area: ' + str(imp_dept))

# convert column now to numeric
df['Area (m2)'] = pd.to_numeric(df['Area (m2)'])

In [ ]:
from datetime import datetime

In [ ]:
#dConverting MonthYear to pandas datetime
df['MonthYear'] = pd.to_datetime(df['MonthYear'],format='%m.%Y', errors='coerce')
# .dt.to_period('m')

In [ ]:
errors = df.loc[df['MonthYear'].isnull()]
errors

In [ ]:
df['MonthYear'][errors.index]

In [ ]:
df['MonthYear'][errors.index].fillna(df['MonthYear'][errors.index-1], inplace=True)
# data['date'] = pd.to_numeric(pd.to_datetime(data['date']))

In [ ]:
df['month'] = df.MonthYear.dt.month
df['month'].unique()

In [ ]:
df.dtypes

# Column Removal and Renaming

In [ ]:
df.columns

In [ ]:
df = df.rename(columns = {'Time index': 'TimeIndex', 'Dept_ID': 'DeptID',
                                  'Dept. Name': 'DeptName',
                                  'Sales units': 'SalesUnits',
                                  'Area (m2)': 'Area','Opening hours': 'OpeningHours'})

In [ ]:
df['StoreID'].value_counts().value_counts()

In [ ]:
df['City'].value_counts().value_counts()

In [ ]:
df['DeptID'].value_counts().value_counts()

In [ ]:
df['DeptName'].value_counts().value_counts()

In [ ]:
df.drop(['StoreID','DeptID','MonthYear'], axis=1, inplace=True)

In [ ]:
df.info()

# Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
# checking the percentage of missing values in each variable
df.isnull().sum()/len(df)*100

In [ ]:
import missingno as msno

# Visualize missing values as a matrix
msno.matrix(df)

In [ ]:
# Visualize the correlation between the number of
# missing values in different columns as a heatmap
msno.heatmap(df)

In [ ]:
# Visualizing the hierarchical relationship
msno.dendrogram(df)

In [ ]:
df.drop(['Customer'], axis=1, inplace=True)

In [ ]:
con_vars = ['HoursOwn', 'HoursLease', 'SalesUnits', 'Turnover', 'Area']
cat_vars = ['TimeIndex', 'Country', 'City',
       'DeptName', 'OpeningHours', 'month']

In [ ]:
df.describe()

# Multiple Imputation of Missing values 

In [ ]:
X = df[con_vars].values
X

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imp = IterativeImputer(max_iter=10, random_state=0)

In [ ]:
imp.fit(X)

In [ ]:
X = imp.transform(X)

In [ ]:
df_con = pd.DataFrame(data=X, columns=con_vars)

In [ ]:
df_con.info()

In [ ]:
# cat_features=[x for x in dataset_train.columns if dataset_train[x].dtype=="object"]
cat_vars = ['TimeIndex', 'Country', 'City',
       'DeptName', 'OpeningHours','month']

In [ ]:
df_cat = df[cat_vars]
df_cat.info()

In [ ]:
# from sklearn import preprocessing
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

ordinal_enc_dict = {}
# encoder = preprocessing.OrdinalEncoder()

# Loop over columns to encode
for columns in cat_vars:
    ordinal_enc_dict[columns] = OrdinalEncoder()
    # Retains only non-null values
    nonulls = np.array(df_cat[columns].dropna())
    # Reshapes the data for encoding
    impute_reshape = nonulls.reshape(-1,1)
    # Encode data
    impute_ordinal = ordinal_enc_dict[columns].fit_transform(impute_reshape)
    # Assign back encoded values to non-null values
    df_cat[columns].loc[df_cat[columns].notnull()] = np.squeeze(impute_ordinal)


In [ ]:
#create a for loop to iterate through each column in the data
df_cat

In [ ]:
# df_cat_imputed = df_cat.copy(deep=True)
# Create MICE imputer
W = df_cat[cat_vars].values
W


In [ ]:
df_cat_imputed = df_cat.copy(deep=True)

In [ ]:
df_cat_imputed.iloc[:, :] = np.round(imp.fit_transform(df_cat_imputed))

In [ ]:
for col in df_cat_imputed:
    reshaped_col = df_cat_imputed[col].values.reshape(-1, 1)
    df_cat_imputed[col] = ordinal_enc_dict[col].inverse_transform(reshaped_col)

In [ ]:
df_cat_imputed[cat_vars]

In [ ]:
# df_cat_imputed = df_cat_imputed.astype(str)

In [ ]:
df_cat_imputed.info()

In [ ]:
df = pd.concat([df_con, df_cat_imputed], axis=1)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# Just a helper module to make visualizations more intuitive
num_to_month={1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'July', 8:'Aug',9:'Sept', 10:'Oct', 11:'Nov',12:'Dec'}
df.month=df.month.map(num_to_month)

In [ ]:
df.head()

In [ ]:
#Let's Check if we have any duplicated row
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

In [ ]:
# df['HoursLease'].unique()

## VISUALIZE AND EXPLORE DATASET

# Visualizing the Probability Density of a continuous variable using distplot  with seaborn kdeplot()

In [ ]:
plt.style.use('fivethirtyeight')
plt.figure(figsize=(20,15))
plt.subplot(421)
sns.distplot(df['HoursOwn'], bins=10, color='blue', label='Histogram of Hours Own')
plt.legend(loc='best')
plt.subplot(422)
sns.distplot(df['HoursLease'], bins=10, color='green', label='Histogram of Hours Lease')
plt.legend(loc='best')
plt.subplot(423)
sns.distplot(df['SalesUnits'], bins=10, color='orange', label='Histogram of Sales units')
plt.legend(loc='best')
plt.subplot(424)
sns.distplot(df['Turnover'], bins=10, color='grey', label='Histogram of Turnover')
plt.legend(loc='best')
plt.subplot(425)
sns.distplot(df['Area'], bins=10, color='red', label='Histogram of Working Area (m2)')
plt.legend(loc='best')
plt.ticklabel_format(style = 'plain', axis = 'x', scilimits=(0,1))
plt.tight_layout()

In [ ]:
Monthly_Sales_Unit = df.groupby(['Country','month']).agg({'SalesUnits':'sum'}).reset_index().rename(columns={'SalesUnits':'Sales_units_amounts'})
Monthly_Sales_Unit.head(10)

## What was the best month for sales? How much was earned that month?

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=Monthly_Sales_Unit, x='Country', y='Sales_units_amounts', hue='month')
plt.title('Monthly_Sales_Unit')
plt.xticks(rotation = 'vertical')
plt.show()

In [ ]:
Monthly_Sales_Unit[Monthly_Sales_Unit['Country']=='Germany']

In [ ]:
Monthly_Sales_Item = df.groupby(['DeptName','month']).agg({'SalesUnits':'sum'}).reset_index().rename(columns={'SalesUnits':'Sales_units_amounts'})
Monthly_Sales_Item.head(10)

## How much sales of each item was earned per month?

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=Monthly_Sales_Item, x='DeptName', y='Sales_units_amounts', hue='month')
plt.title('Monthly_Sales_Item')
plt.xticks(rotation = 'vertical')
plt.show()

In [ ]:
Monthly_Sales_Item[Monthly_Sales_Item['DeptName']=='Admin']

In [ ]:
Monthly_Sales_Item[Monthly_Sales_Item['DeptName']=='Checkout']

In [ ]:
Sales_Item_in_city = df.groupby(['City']).agg({'SalesUnits':'sum'}).reset_index().rename(columns={'SalesUnits':'Sales_units_in_citys'})
Sales_Item_in_city.head()

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=Sales_Item_in_city, x='City', y='Sales_units_in_citys')
plt.title('Sales_Item_in_city')
plt.xticks(rotation = 'vertical')
plt.show()

# Visualizing the Scater Plot Density of a continuous variables 

In [ ]:
con_vars = ['HoursOwn', 'HoursLease', 'SalesUnits', 'Turnover', 'Area']
df1 = df[con_vars]
df1 = pd.DataFrame(data=df1, columns= con_vars)

In [ ]:
# Visualize the relationship between variables using scatter plots
import plotly.express as px
fig = px.scatter(df1,
                x='HoursOwn', y='HoursLease', color='SalesUnits')# fill color by months
fig.update_layout(title_text='SalesUnits',
                  width=810,
                  height=510
                 )
fig.show()

In [ ]:
# Visualize the relationship between variables using pairplots

fig = px.scatter_matrix(df1,
                        dimensions=df1.columns[:], color='SalesUnits')# fill color by months
fig.update_layout(title_text='SalesUnits',
                  width=1100,
                  height=1100
                 )
fig.show()

# Outlier Detection Using Box Plot

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df, x = 'DeptName', y = 'SalesUnits')
fig.update_layout(title_text='SalesUnits', width=910, height=610)
fig.show()

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df, x = 'DeptName', y = 'HoursOwn')
fig.update_layout(title_text='HoursOwn', width=910, height=610)
fig.show()

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df, x = 'DeptName', y = 'HoursLease')
fig.update_layout(title_text='HoursLease', width=910, height=610)
fig.show()

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df, x = 'DeptName', y = 'Turnover')
fig.update_layout(title_text='Turnover', width=910, height=610)
fig.show()

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df, x = 'DeptName', y = 'Area')
fig.update_layout(title_text='Area', width=910, height=610)
fig.show()

# Removing Outliers

In [ ]:
# First we will calculate IQR
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
print(IQR)

In [ ]:
#filter out the outliers by keeping only valid values
def remove_outlier_IQR(df):
    Q1=df.quantile(0.25)
    Q3=df.quantile(0.75)
    IQR=Q3-Q1
    df_final=df[~((df<(Q1-1.5*IQR)) | (df>(Q3+1.5*IQR)))]
    return df_final

In [ ]:
df_outlier_removed=remove_outlier_IQR(df.HoursOwn)
df_outlier_removed=pd.DataFrame(df_outlier_removed)
ind_diff=df.index.difference(df_outlier_removed.index)

for i in range(0, len(ind_diff),1):
    df_final=df.drop([ind_diff[i]])
    df=df_final

In [ ]:
df_outlier_removed=remove_outlier_IQR(df.HoursLease)
df_outlier_removed=pd.DataFrame(df_outlier_removed)
ind_diff=df.index.difference(df_outlier_removed.index)

for i in range(0, len(ind_diff),1):
    df_final=df.drop([ind_diff[i]])
    df=df_final

In [ ]:
df_outlier_removed=remove_outlier_IQR(df.SalesUnits)
df_outlier_removed=pd.DataFrame(df_outlier_removed)
ind_diff=df.index.difference(df_outlier_removed.index)

for i in range(0, len(ind_diff),1):
    df_final=df.drop([ind_diff[i]])
    df=df_final

In [ ]:
df_outlier_removed=remove_outlier_IQR(df.Turnover)
df_outlier_removed=pd.DataFrame(df_outlier_removed)
ind_diff=df.index.difference(df_outlier_removed.index)

for i in range(0, len(ind_diff),1):
    df_final=df.drop([ind_diff[i]])
    df=df_final

In [ ]:
df_outlier_removed=remove_outlier_IQR(df.Area)
df_outlier_removed=pd.DataFrame(df_outlier_removed)
ind_diff=df.index.difference(df_outlier_removed.index)

for i in range(0, len(ind_diff),1):
    df_final=df.drop([ind_diff[i]])
    df=df_final
df.reset_index(drop=True, inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

# correlation between numeric features

In [ ]:
corr = df[con_vars].corr()
corr

In [ ]:
df.isnull().sum()

In [ ]:
#Let's investigate correlation between each values
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='RdBu')

In [ ]:
#Correlation matrix (heatmap style)
#correlation matrix
corrmat = df.corr()
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat, vmax=.8, square=True);


In [ ]:
print(corr['SalesUnits'].sort_values(ascending=False))

In [ ]:
df['HoursLease'].unique()

In [ ]:
df.drop(['HoursLease'], axis=1, inplace=True)
con_vars.remove('HoursLease')

# Enconding Categorical Variables

In [ ]:
df.columns

In [ ]:
# Convert categorical variables to string
cat_vars = ['TimeIndex', 'Country', 'City',
       'DeptName', 'OpeningHours', 'month']
df[cat_vars] = df[cat_vars].astype(str)
df.info()

# Checking unique features and data pattern for categorical variables

In [ ]:
df[cat_vars].describe(include=['O'])

In [ ]:
# One hot encoding - to convert categorical data to continuous
df[cat_vars] = df[cat_vars].astype(str)
df2 = pd.get_dummies(df)
df2.head(5)

# RobustScaler Normalization

In [ ]:
# Normalizing
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
df2[con_vars]=scaler.fit_transform(df2[con_vars])
df2.head(5)

In [ ]:
plt.style.use('fivethirtyeight')
plt.figure(figsize=(20,15))
plt.subplot(421)
sns.distplot(df2['HoursOwn'], bins=10, color='blue', label='Histogram of Hours Own')
plt.legend(loc='best')
plt.subplot(422)
sns.distplot(df2['SalesUnits'], bins=10, color='orange', label='Histogram of Sales units')
plt.legend(loc='best')
plt.subplot(423)
sns.distplot(df2['Turnover'], bins=10, color='grey', label='Histogram of Turnover')
plt.legend(loc='best')
plt.subplot(424)
sns.distplot(df2['Area'], bins=10, color='red', label='Histogram of Working Area (m2)')
plt.legend(loc='best')
plt.ticklabel_format(style = 'plain', axis = 'x', scilimits=(0,1))
plt.tight_layout()

In [ ]:
df3 = df2[con_vars]
df3 = pd.DataFrame(data=df3, columns= con_vars)

In [ ]:
# Visualize the relationship between variables using pairplots
import plotly.express as px
fig = px.scatter(df3,
                x='HoursOwn', y='Turnover', color='SalesUnits')# fill color by months
fig.update_layout(title_text='SalesUnits',
                  width=810,
                  height=510
                 )
fig.show()

In [ ]:
# Visualize the relationship between variables using pairplots

fig = px.scatter_matrix(df3,
                        dimensions=df3.columns[:], color='SalesUnits')# fill color by months
fig.update_layout(title_text='SalesUnits',
                  width=1100,
                  height=1100
                 )
fig.show()

In [ ]:
df3['DeptName'] = df['DeptName']


In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(df3, x = 'DeptName', y = 'SalesUnits')
fig.update_layout(title_text='SalesUnits', width=910, height=610)
fig.show()

fig=px.box(df3, x = 'DeptName', y = 'HoursOwn')
fig.update_layout(title_text='HoursOwn', width=910, height=610)
fig.show()

fig=px.box(df3, x = 'DeptName', y = 'Turnover')
fig.update_layout(title_text='Turnover', width=910, height=610)
fig.show()

fig=px.box(df3, x = 'DeptName', y = 'Area')
fig.update_layout(title_text='Area', width=910, height=610)
fig.show()

In [ ]:
corr = df3.corr()
corr

In [ ]:
#Let's investigate correlation between each values
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='RdBu')

In [ ]:
#Correlation matrix (heatmap style)
#correlation matrix
corrmat = df3.corr()
f, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corrmat, vmax=.8, square=True)

In [ ]:
print(corr['SalesUnits'].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(8, 6))
heatmap = sns.heatmap(df3.corr()[['SalesUnits']].sort_values(by='SalesUnits', ascending=False), vmin=-1, vmax=1, annot=True, cmap='YlGnBu')
heatmap.set_title('Features Correlating with Sales units', fontdict={'fontsize':18}, pad=16)

# PCA

In [ ]:
# Define the final test sets
y = df2['SalesUnits']


df4 = df2.drop(['SalesUnits'], axis = 1)

In [ ]:
np.any(np.isnan(y))

In [ ]:
#create 16 component PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=28)
pca_comp = pca.fit_transform(df4)

In [ ]:
explained_variane = pca.explained_variance_ratio_

In [ ]:
explained_variane

In [ ]:
explained_variane.sum()

In [ ]:
pca_df = pd.DataFrame(pca_comp)
pca_df = pca_df.add_prefix('pca_')

In [ ]:
pca_df

In [ ]:
corr = pca_df.corr()
corr

In [ ]:
#Correlation matrix (heatmap style)
#correlation matrix
corrmat = pca_df.corr()
f, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corrmat, vmax=.8, square=True)

In [ ]:
#add the target to new dataframe
pca_df = pd.concat([pca_df, pd.DataFrame({'SalesUnits':y})], axis=1)

In [ ]:
pca_df.head()

In [ ]:
plt.style.use('fivethirtyeight')
plt.figure(figsize=(20,15))
plt.subplot(421)
sns.distplot(pca_df['pca_0'], bins=10, color='blue', label='Histogram of Hours Own')
plt.legend(loc='best')
plt.subplot(422)
sns.distplot(pca_df['pca_1'], bins=10, color='green', label='Histogram of Hours Lease')
plt.legend(loc='best')
plt.ticklabel_format(style = 'plain', axis = 'x', scilimits=(0,1))
plt.tight_layout()

In [ ]:
# Visualize the relationship between variables using pairplots
import plotly.express as px
fig = px.scatter(pca_df,
                x='pca_0', y='pca_1', color='SalesUnits')# fill color by months
fig.update_layout(title_text='SalesUnits',
                  width=810,
                  height=510
                 )
fig.show()

In [ ]:
pca_df['DeptName'] = df['DeptName']

In [ ]:
# Draw a vertical boxplot grouped
# by a categorical variable:
fig=px.box(pca_df, x= 'DeptName', y = 'pca_0')
fig.update_layout(title_text='pca_0', width=910, height=610)
fig.show()

fig=px.box(pca_df, x= 'DeptName', y = 'pca_1')
fig.update_layout(title_text='pca_1', width=910, height=610)
fig.show()

In [ ]:
pca_df = pca_df.drop(['DeptName'], axis=1)
corr = pca_df.corr()
print(corr['SalesUnits'].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(8, 6))
heatmap = sns.heatmap(pca_df.corr()[['SalesUnits']].sort_values(by='SalesUnits', ascending=False), vmin=-1, vmax=1, annot=True, cmap='YlGnBu')
heatmap.set_title('Features Correlating with Sales units', fontdict={'fontsize':18}, pad=16)

# Splitting the data-set into Training and Test Set

In [ ]:
#Split in 80% train and 20% test set


train, test = train_test_split(pca_df, test_size = 0.2, random_state= 1984)

In [ ]:
# Define the final train and test sets
y_train = train.SalesUnits
y_test = test.SalesUnits

x_train = train.drop(['SalesUnits'], axis = 1)
x_test = test.drop(['SalesUnits'], axis = 1)

In [ ]:
x_train =np.array(x_train)
x_test = np.array(x_test)

y_train = np.array(y_train)
y_test = np.array(y_test)

# XGBoost Model

In [ ]:
# create an xgboost regression model
import xgboost as xgb

model = xgb.XGBRegressor()

# Model Training

In [ ]:
model.fit(x_train, y_train)

# Model prediction

In [ ]:
y_pred = model.predict(x_test)

# Evaluating the result

# R-Squared

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(y_test, y_pred)

# Root Mean Squared Error

In [ ]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test, y_pred, squared=False)